# Phase 3 · Notebook 03 — Lite vs Pro Side-by-Side

Both pipelines on the same documents, with the differences called out:

  * **Lite** — DIRECT-only redaction. Quasi-identifiers untouched.
  * **Pro**  — DIRECT + iterate-until-safe QUASI generalization.

The point isn't to declare a winner — both have valid use cases — but to make the trade-off concrete.

---


## Setup


In [1]:
import sys
sys.path.insert(0, "../../src")

from anonymisation.data import load_tab
from anonymisation.mapping import SPACY_TO_TAB
from anonymisation.pipeline import LitePipeline, ProPipeline, MosaicScorer

import spacy
nlp = spacy.load("en_core_web_trf")

def spacy_predictor(text):
    doc = nlp(text)
    return [(e.start_char, e.end_char, SPACY_TO_TAB[e.label_], e.text)
            for e in doc.ents if e.label_ in SPACY_TO_TAB]

ds = load_tab()
scorer = MosaicScorer.from_tab(list(ds["test"]))

lite = LitePipeline(ner_provider=spacy_predictor)
pro  = ProPipeline(ner_provider=spacy_predictor, scorer=scorer, k_target=5)


## A few targeted examples

Each input is chosen to highlight a different aspect of the contrast.


In [2]:
samples = [
    # 1 — single-sentence, lots of quasi-identifiers
    ("The applicant is a 47-year-old Bulgarian national living in Plovdiv, "
     "employed as a nurse since 2010."),
    # 2 — mostly direct identifiers
    ("Maria Petrova was represented by Sofia District Court in Application no. 12345/67."),
    # 3 — a sentence with rare demographics that combine into a fingerprint
    ("The applicant is a Roma asylum-seeker, mother of three, born in 1977 "
     "in Burgas, who worked as a translator for the Bulgarian army."),
]

for i, text in enumerate(samples, 1):
    print(f"\n══════════════════════ SAMPLE {i} ══════════════════════")
    print(f"INPUT:\n  {text}\n")

    lr = lite(text)
    print(f"LITE → {lr.redacted_text}")

    pr = pro(text)
    print(f"PRO  → {pr.redacted_text}")
    print(f"       (k: {pr.mosaic_risk_initial} → {pr.mosaic_risk_final}, "
          f"iters: {pr.iterations_used}, converged: {pr.converged})")



══════════════════════ SAMPLE 1 ══════════════════════
INPUT:
  The applicant is a 47-year-old Bulgarian national living in Plovdiv, employed as a nurse since 2010.

LITE → The applicant is a 47-year-old Bulgarian national living in Plovdiv, employed as a nurse since 2010.
PRO  → The applicant is a [DATETIME] [DEM] national living in [LOC], employed as a nurse since [DATETIME].
       (k: 1 → 555, iters: 3, converged: True)

══════════════════════ SAMPLE 2 ══════════════════════
INPUT:
  Maria Petrova was represented by Sofia District Court in Application no. 12345/67.

LITE → [PERSON] was represented by Sofia District Court in [CODE].
PRO  → [PERSON] was represented by [LOC] District Court in [CODE].
       (k: 1 → 555, iters: 3, converged: True)

══════════════════════ SAMPLE 3 ══════════════════════
INPUT:
  The applicant is a Roma asylum-seeker, mother of three, born in 1977 in Burgas, who worked as a translator for the Bulgarian army.

LITE → The applicant is a Roma asylum-seeker

/Users/williamcatt/Documents/Projects/Data Anonymisation/legal-anon-env/lib/python3.11/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


## What's in the difference

For a typical legal sentence the diff between Lite and Pro looks like this:

| Field | Lite | Pro |
|---|---|---|
| Maria Petrova | `[PERSON]` | `[PERSON]` |
| 47-year-old | `47-year-old` | `about 50` → `in their 40s` → `[QUANTITY]` |
| Bulgarian | `Bulgarian` | `European` → `[DEM]` |
| Plovdiv | `Plovdiv` | `Bulgaria` → `Europe` → `[LOC]` |
| 2010 | `2010` | `2010` → `2010s` → `[DATETIME]` |

How far each QUASI advances depends on what the mosaic scorer says is necessary — Pro stops at the first level where the document's residual fingerprint reaches k_target.


## Picking between the two

A practical rubric I'd give a buyer:

> **Use Lite when:** the threat is "an LLM provider could log our prompts" or "a curious employee at our SaaS vendor". The right tool is to strip the obvious DIRECT identifiers and accept that quasi-identifiers stay.
>
> **Use Pro when:** the threat is "a determined adversary with access to public records and the redacted document". Common in regulated industries, employee-data work, and high-profile matters where redaction is a compliance requirement rather than a hygiene step.

Both pipelines produce the same shape of audit log, so a firm can A/B them on the same documents and pick a default per matter type.
